# Sizing the sw_tgate Switch

Adapted from `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, which is the
overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1


A switch carries no bias current and its gate sits at a rail, so gm/ID is not the knob
here. What matters is the on resistance, and section 3.5 of the Analog Circuit Design course
reads it straight out of the same tables as `Ron = 1/GDS` in triode at a small `VDS`.

`Ron` in a transmission gate is not one number. The NMOS gate sits at `VDD` with its bulk at
`VSS`, the PMOS the other way round, so as the switched potential moves between the rails each
device loses gate drive at one end while its body effect grows. The NMOS is the good switch
near `VSS`, the PMOS near `VDD`, and only the parallel pair covers the range. The number a
budget has to meet is the worst case over that sweep.


In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Supply, the triode probe and the switched capacitance this cell drives
VDD   = 1.5
VDS   = 0.01          # 10 mV, the Ron lookup point
CAP   = 6.4e-14            # F, what this switch charges
CELL  = 'sw_tgate'
DATA  = Path('../../../macros/inverter/scripts/sizing/data')
SCH   = Path('../schematic') / (CELL + '.sch')

### Read the sizes out of the schematic

The schematic is the source of truth, so nothing here restates it. This also checks the two
invariants the topology rests on: one channel length throughout, and dummies at half the width
of the device they shadow, which is what cancels the injected charge.

In [3]:
def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'\{name=(M\w+)\n(.*?)\n\}', txt, re.S):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(2), re.M))
        dev[m.group(1)] = (float(a['w'].rstrip('u')), float(a['l'].rstrip('u')))
    (wn, ln), (wp, _) = dev['MN'], dev['MP']
    assert len({l for _, l in dev.values()}) == 1, 'mixed channel lengths'
    for d, ref in (('MND1', wn), ('MND2', wn), ('MPD1', wp), ('MPD2', wp)):
        assert abs(dev[d][0] - ref / 2) < 1e-9, d + ' is not half of its device'
    return wn, wp, ln

WN, WP, L = read_sizes(SCH)
print(f'{CELL}: Wn {WN} um, Wp {WP} um, L {L} um, Wp/Wn = {WP/WN:g}')

sw_tgate: Wn 2.0 um, Wp 4.0 um, L 1.0 um, Wp/Wn = 2


### Conductance against the switched potential

A vector `VGS` against a vector `VSB` comes back as a matrix, so the diagonal is the pairing
that belongs together. Dividing by the width the table was swept at gives S/um.

Where a device is fully off the interpolated `GDS` goes slightly negative, a handful of points
at the sweep ends. Clamped at zero an off device simply stops contributing, which is the
physical answer, where a negative resistance would subtract from the parallel combination.

In [4]:
nmos = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pmos = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
VS = np.arange(0.2, 1.41, 0.01)        # the course sweep, clear of the table edges

gds_n = np.diag(nmos.lookup('GDS', L=L, VGS=VDD - VS, VDS=VDS, VSB=VS))
gds_p = np.diag(pmos.lookup('GDS', L=L, VGS=np.abs(VS), VDS=VDS, VSB=np.abs(VS - VDD)))
gw_n = np.maximum(gds_n, 0) / nmos['W']       # S/um
gw_p = np.maximum(gds_p, 0) / pmos['W']

### The cell as drawn

In [5]:
ron = 1 / (WN * gw_n + WP * gw_p)
k   = int(np.argmax(ron))

print(f'worst case      {ron[k]:8.0f} ohm at Vs = {VS[k]:.2f} V')
print(f'at the sweep    {ron[0]:8.0f} ohm at {VS[0]:.2f} V, {ron[-1]:.0f} ohm at {VS[-1]:.2f} V')
print(f'  nmos alone    {1/(WN*gw_n[k]):8.0f} ohm there')
print(f'  pmos alone    {1/(WP*gw_p[k]):8.0f} ohm there')
print(f'Ron*Csw         {ron[k]*CAP*1e9:8.3f} ns with {CAP*1e15:.0f} fF, against Ts/2 = 488 ns')

worst case          4044 ohm at Vs = 1.13 V
at the sweep        1460 ohm at 0.20 V, 3190 ohm at 1.40 V
  nmos alone       60162 ohm there
  pmos alone        4336 ohm there
Ron*Csw            0.259 ns with 64 fF, against Ts/2 = 488 ns


### Sizing for a target

`Wp = 3*Wn` is the house ratio for this process. With `Wp` tied to `Wn` the total conductance
is linear in `Wn`, so the width inverts directly from the worst-case point.

In [6]:
RON_TARGET = 2e3          # ohm
WP_OVER_WN = 3.0

g_tg = gw_n + WP_OVER_WN * gw_p           # S/um, referred to Wn
wn   = 1 / (RON_TARGET * g_tg.min())
print(f'for Ron <= {RON_TARGET:.0f} ohm at L = {L} um: Wn {wn:.3f} um, Wp {WP_OVER_WN*wn:.3f} um')

for Ron <= 2000 ohm at L = 1.0 um: Wn 2.780 um, Wp 8.340 um


### Why this cell is not at Wp = 3*Wn

It is drawn at 2, and the difference was measured rather than assumed. This switch sits on the
floating virtual ground, so its off-state subthreshold leakage is what sets the integrator
leak, and 1.5 times the PMOS width is 1.5 times the leak. On the `sc_int` step response,
where the impulse should be -250 mV and hold, going to 3 takes the droop from 1.1 % to 20 %.

What the ratio buys is on resistance, and the cell above shows on resistance was never the
binding constraint: a fraction of a nanosecond against a 488 ns half period, with the
amplifier's own 16.8 ns dominating. So this switch keeps the narrow PMOS and gives up nothing
that matters. `sw_tgate_cmfb` is not on that node and does use 3.